# Geopolitical Events & Bitcoin Price — LSTM Predictive Model

**Part 1 · Quantitative Phase**  

This notebook implements a univariate LSTM model trained on daily Bitcoin (BTC/USD) closing prices from the Bitstamp exchange (2012–2014), with inference over the full year 2015. The primary goal is **not** accurate price forecasting per se, but rather the identification of dates where model prediction error is anomalously large — which serve as empirical candidates for geopolitical event investigation in Phase 2 of the research.

---

**Workflow overview:**
1. Data loading and preprocessing  
2. Temporal train/test/validation split  
3. MinMax normalisation and sliding-window transformation  
4. Baseline LSTM — architecture, training, evaluation  
5. Fine-tuned LSTM (stacked, deeper architecture)  
6. Hyperparameter grid search  
7. Final model — full 2015 inference and residual analysis


## 1 · Imports

In [ ]:
import math

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow version: {tf.__version__}")
print("All libraries imported successfully.")


## 2 · Data Loading & Preprocessing

The dataset is `bitstamp_daily_final.csv`, containing daily OHLCV data sourced from the Bitstamp exchange.  
Update the path below to match your local environment or clone from the project repository.


### 2.1 · Configuration

In [ ]:
# ── Global configuration ─────────────────────────────────────────────────────
DATA_PATH  = "bitstamp_daily_final.csv"
LOOK_BACK  = 60    # sliding window size (days)
TARGET_COL = "Close"


### 2.2 · Load & inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
df["DateTime"] = pd.to_datetime(df["DateTime"])
df = df.set_index("DateTime").sort_index()

print(f"Dataset loaded. Total trading days: {len(df)}")
print(df.head())


### 2.3 · Temporal split

A **strictly temporal** split is applied to prevent data leakage:

| Subset | Period | Purpose |
|--------|--------|---------|
| Train | 2012-01-01 → 2014-09-30 | Model learning |
| Test | 2014-10-01 → 2014-12-31 | Validation during training |
| Validation | 2015-01-01 → 2015-12-31 | Out-of-sample inference (Phase 2) |

The 2012–2015 window is deliberately chosen: it covers Bitcoin's early adoption period, before the market matured, and aligns with the geopolitical events under investigation.


In [ ]:
TRAIN_END   = "2014-09-30"
TEST_START  = "2014-10-01"
TEST_END    = "2014-12-31"
VAL_START   = "2015-01-01"
VAL_END     = "2015-12-31"

train_df = df.loc[:TRAIN_END].copy()
test_df  = df.loc[TEST_START:TEST_END].copy()
val_df   = df.loc[VAL_START:VAL_END].copy()

print("--- Temporal split ---")
print(f"Train  (2012 → Sep/2014) : {len(train_df):>5} days")
print(f"Test   (Oct → Dec/2014) : {len(test_df):>5} days")
print(f"Val.   (2015)            : {len(val_df):>5} days")


### 2.4 · Normalisation

`MinMaxScaler` is fitted **exclusively on the training set** to avoid information leakage from future data into the scaling parameters.


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_df[[TARGET_COL]])
test_scaled  = scaler.transform(test_df[[TARGET_COL]])
val_scaled   = scaler.transform(val_df[[TARGET_COL]])


### 2.5 · Sliding-window transformation

The time series is reshaped into supervised learning sequences.  
Each sample `X[i]` contains `LOOK_BACK` consecutive closing prices, and `y[i]` is the next day's price.
The resulting shape matches the LSTM expected input: `(samples, timesteps, features)`.


In [ ]:
def create_dataset(dataset: np.ndarray, look_back: int):
    """Convert a scaled 1-D array into (X, y) supervised sequences."""
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i : i + look_back, 0])
        y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(y)


X_train, y_train = create_dataset(train_scaled, LOOK_BACK)
X_test,  y_test  = create_dataset(test_scaled,  LOOK_BACK)
X_val,   y_val   = create_dataset(val_scaled,   LOOK_BACK)

# Reshape to 3-D tensor: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],   X_test.shape[1],  1)
X_val   = X_val.reshape(X_val.shape[0],     X_val.shape[1],   1)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print(f"X_val   shape : {X_val.shape}")


## 3 · Baseline LSTM

A single-layer LSTM with 50 units serves as the baseline. Its purpose is to:
- confirm that the loss decreases meaningfully across epochs; and
- establish a reference error floor against which the fine-tuned model is compared.


### 3.1 · Architecture

In [ ]:
def build_baseline(input_shape: tuple) -> Sequential:
    """
    Lightweight single-layer LSTM baseline.

    Architecture:
        LSTM(50)  → Dropout(0.2)  → Dense(1)
    """
    model = Sequential([
        LSTM(units=50, return_sequences=False, input_shape=input_shape),
        Dropout(0.2),
        Dense(units=1),
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model


model = build_baseline((X_train.shape[1], 1))
model.summary()


### 3.2 · Training

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1,
)


### 3.3 · Learning curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"],     label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Baseline — Learning Curve (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 3.4 · Evaluation on test set (Oct–Dec 2014)

In [ ]:
# Generate predictions and invert normalisation
predicted_baseline = model.predict(X_test)
predicted_baseline_real = scaler.inverse_transform(predicted_baseline)
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))

# Compute error metrics
rmse_baseline = math.sqrt(mean_squared_error(y_test_real, predicted_baseline_real))
mae_baseline  = mean_absolute_error(y_test_real, predicted_baseline_real)
mean_price    = np.mean(y_test_real)

print("--- Baseline Performance (Oct–Dec 2014) ---")
print(f"RMSE : ${rmse_baseline:.2f}  ({rmse_baseline / mean_price * 100:.2f}% of mean price)")
print(f"MAE  : ${mae_baseline:.2f}  ({mae_baseline  / mean_price * 100:.2f}% of mean price)")


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test_real,            color="steelblue", label="Actual Price",      linewidth=1.5)
plt.plot(predicted_baseline_real, color="tomato",    label="Baseline Forecast", linewidth=1.5, linestyle="--")
plt.title("Baseline LSTM — BTC/USD Forecast vs Actual (Oct–Dec 2014)")
plt.xlabel("Days (test window)")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4 · Fine-Tuned LSTM

A deeper, stacked architecture is used to improve on the baseline. Key changes:
- Two LSTM layers (128 → 64 units) with `return_sequences=True` on the first layer;
- An additional `Dense(25)` intermediate layer for refinement;
- Explicit `Adam` learning rate (`lr=0.001`);
- `ModelCheckpoint` to persist the best epoch weights.


### 4.1 · Architecture

In [ ]:
def build_tuned_model(input_shape: tuple) -> Sequential:
    """
    Stacked two-layer LSTM with intermediate dense layer.

    Architecture:
        LSTM(128, return_seq=True) → Dropout(0.2)
        → LSTM(64)                → Dropout(0.2)
        → Dense(25)               → Dense(1)
    """
    model = Sequential([
        LSTM(units=128, return_sequences=True,  input_shape=input_shape),
        Dropout(0.2),
        LSTM(units=64,  return_sequences=False),
        Dropout(0.2),
        Dense(units=25),
        Dense(units=1),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mean_squared_error")
    return model


model_tuned = build_tuned_model((X_train.shape[1], 1))
model_tuned.summary()


### 4.2 · Training

In [ ]:
checkpoint = ModelCheckpoint(
    "best_btc_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=0,
)

history_v2 = model_tuned.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, checkpoint],
    verbose=1,
)


### 4.3 · Evaluation on test set (Oct–Dec 2014)

In [ ]:
best_model = tf.keras.models.load_model("best_btc_model.keras")

# Learning curve
plt.figure(figsize=(10, 5))
plt.plot(history_v2.history["loss"],     label="Train Loss")
plt.plot(history_v2.history["val_loss"], label="Validation Loss")
plt.title("Fine-Tuned Model — Learning Curve (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
predictions_v2 = best_model.predict(X_test)
predictions_v2_real = scaler.inverse_transform(predictions_v2)
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse_v2 = math.sqrt(mean_squared_error(y_test_real, predictions_v2_real))
mae_v2  = mean_absolute_error(y_test_real, predictions_v2_real)
improvement = ((mae_baseline - mae_v2) / mae_baseline) * 100

print("--- Fine-Tuned Model Performance (Oct–Dec 2014) ---")
print(f"RMSE     : ${rmse_v2:.2f}")
print(f"MAE      : ${mae_v2:.2f}")
print(f"Baseline MAE : ${mae_baseline:.2f}")
print(f"Improvement  : {improvement:.1f}%")


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test_real,        color="black",  label="Actual Price",      linewidth=1.5)
plt.plot(predictions_v2_real, color="green",  label="Fine-Tuned Forecast", linewidth=1.5, linestyle="--")
plt.title("Fine-Tuned LSTM — BTC/USD Forecast vs Actual (Oct–Dec 2014)")
plt.xlabel("Days (test window)")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5 · Hyperparameter Grid Search

A lightweight grid search explores the cross product of two hyperparameters:
- `look_back` ∈ {60, 90} days
- `neurons` ∈ {50, 100}

Models are trained for a maximum of 20 epochs with early stopping (`patience=5`). MAE on the test set is used as the selection criterion.


In [ ]:
# Merge train + test for the sliding window to cover both periods
scaled_data = np.concatenate((train_scaled, test_scaled), axis=0)

look_back_options = [60, 90]
neurons_options   = [50, 100]
gs_results        = []

def _create_dataset(dataset, look_back):
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i : i + look_back, 0])
        y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(y)


print(f"Grid Search — {len(look_back_options) * len(neurons_options)} combinations to evaluate\n")

for lb in look_back_options:
    for neurons in neurons_options:
        config_name = f"Win{lb}_Neur{neurons}"
        print(f"  Testing: {config_name} ...", end=" ")

        X_all, y_all = _create_dataset(scaled_data, lb)
        X_all = X_all.reshape(X_all.shape[0], X_all.shape[1], 1)

        # Align split index with the original train boundary
        split_idx = max(len(train_scaled) - lb, 0)

        X_tr, y_tr = X_all[:split_idx], y_all[:split_idx]
        X_te, y_te = X_all[split_idx:],  y_all[split_idx:]

        m = Sequential([
            LSTM(units=neurons, return_sequences=False, input_shape=(lb, 1)),
            Dropout(0.2),
            Dense(1),
        ])
        m.compile(optimizer="adam", loss="mean_squared_error")

        es_gs = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        m.fit(X_tr, y_tr, epochs=20, batch_size=32,
              validation_data=(X_te, y_te), callbacks=[es_gs], verbose=0)

        preds_real = scaler.inverse_transform(m.predict(X_te, verbose=0))
        y_te_real  = scaler.inverse_transform(y_te.reshape(-1, 1))
        mae        = mean_absolute_error(y_te_real, preds_real)

        print(f"MAE = ${mae:.2f}")
        gs_results.append({"Config": config_name, "MAE": mae, "LookBack": lb, "Neurons": neurons})

results_df  = pd.DataFrame(gs_results).sort_values(by="MAE").reset_index(drop=True)
best_config = results_df.iloc[0]

print("\n--- Grid Search Ranking ---")
print(results_df.to_string(index=False))
print(f"\nBest configuration : {best_config['Config']}  (MAE = ${best_config['MAE']:.2f})")


## 6 · Final Model — Full 2015 Inference

Using the champion hyperparameters from the grid search, a final model is trained on the full 2012–2014 dataset (train + test merged) and used to generate out-of-sample predictions for every trading day in 2015.

The residuals — the absolute difference between actual and predicted prices — serve as the primary signal: **unusually large errors are hypothesised to coincide with geopolitically significant events**, which are investigated qualitatively in Phase 2.


### 6.1 · Final training data preparation

In [ ]:
# Merge train and test into a single training block (full 2012–2014)
X_final_train = np.concatenate((X_train, X_test), axis=0)
y_final_train = np.concatenate((y_train, y_test), axis=0)

print("Final data preparation complete.")
print(f"  Training samples (2012–2014): {X_final_train.shape[0]:>5}")
print(f"  Inference samples (2015)    : {X_val.shape[0]:>5}")


### 6.2 · Training the final model

In [ ]:
model_final = Sequential([
    # Remember to adjust units (neurons) and input_shape if look_back was changed in the grid search
    LSTM(units=100, return_sequences=False, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    Dense(1),
])
model_final.compile(optimizer="adam", loss="mean_squared_error")

history_final = model_final.fit(
    X_final_train, y_final_train,
    epochs=50,
    batch_size=32,
    verbose=1,
)
print("Final model training complete.")


### 6.3 · 2015 inference

In [ ]:
predicted_2015       = model_final.predict(X_val)
predicted_2015_real  = scaler.inverse_transform(predicted_2015)
y_val_real           = scaler.inverse_transform(y_val.reshape(-1, 1))

mae_final  = mean_absolute_error(y_val_real, predicted_2015_real)
rmse_final = math.sqrt(mean_squared_error(y_val_real, predicted_2015_real))

print("--- Final Model Performance (2015 — out-of-sample) ---")
print(f"MAE  : ${mae_final:.2f}")
print(f"RMSE : ${rmse_final:.2f}")


### 6.4 · Residual analysis

In [ ]:
residuals = np.abs(y_val_real - predicted_2015_real)

# Dates corresponding to X_val (offset by LOOK_BACK due to windowing)
dates_2015 = val_df.index[LOOK_BACK:]

analysis_df = pd.DataFrame({
    "Date":             dates_2015,
    "Actual_Price":     y_val_real.flatten(),
    "Predicted_Price":  predicted_2015_real.flatten(),
    "Abs_Error":        residuals.flatten(),
    "Pct_Error":        (residuals.flatten() / y_val_real.flatten()) * 100,
})


### 6.5 · Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# — Panel 1: Actual vs. Predicted ——————————————————————————————————————————
axes[0].plot(dates_2015, y_val_real,          color="black",   linewidth=1.5, label="Actual (BTC/USD)")
axes[0].plot(dates_2015, predicted_2015_real,  color="#00CC96", linewidth=1.5, linestyle="--",
             label="Predicted (LSTM Baseline)")
axes[0].set_title(f"Bitcoin Price Forecast — 2015  (MAE: ${mae_final:.2f})", fontsize=14)
axes[0].set_ylabel("Price (USD)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b/%Y"))

# — Panel 2: Residuals ————————————————————————————————————————————————————
axes[1].bar(dates_2015, analysis_df["Abs_Error"], color="#EF553B", alpha=0.7,
            label="Prediction Error (Residual)")
axes[1].set_title("Residual Magnitude — Where Did the Model Fail? (Candidates for External Event Investigation)",
                  fontsize=13)
axes[1].set_ylabel("Absolute Error (USD)")
axes[1].set_xlabel("Date")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b/%Y"))

plt.tight_layout()
plt.show()


### 6.6 · Top outlier dates for Phase 2 investigation

In [ ]:
top_errors = (
    analysis_df
    .sort_values(by="Abs_Error", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("======  TOP-10 DATES FOR GEOPOLITICAL INVESTIGATION (Phase 2)  ======")
print("Suggested query: 'Bitcoin news [date]' to identify potential causal events.\n")
print(top_errors[["Date", "Actual_Price", "Predicted_Price", "Abs_Error", "Pct_Error"]].to_string(index=False))


---

## 7 · Phase 2 Output Pipeline

This section processes the `analysis_df` produced in Section 6 to generate two CSV files consumed by `Geo-BTC-Phase2_Research.ipynb`:

- `analysis_residuals_2015.csv` — full 302-day residual history with outlier flags
- `investigation_periods.csv`  — asymmetric research windows around each outlier date

### 7.1 · Outlier detection (2σ rule)

The directional error (`Actual − Predicted`) is computed for each day.  
A day is flagged as an outlier if its error falls outside the **μ ± 2σ** band.

In [ ]:
# ── 1. Directional error ──────────────────────────────────────────────────────
analysis_df["Directional_Error"] = analysis_df["Actual_Price"] - analysis_df["Predicted_Price"]

errors = analysis_df["Directional_Error"].values
mean_err = np.mean(errors)
std_err = np.std(errors)
upper_thr = mean_err + 2 * std_err
lower_thr = mean_err - 2 * std_err

print("--- Outlier Detection (2σ Rule) ---")
print(f"μ  (mean error) : {mean_err:.4f} USD")
print(f"σ  (std dev)    : {std_err:.4f} USD")
print(f"+2σ threshold   : {upper_thr:.4f} USD")
print(f"-2σ threshold   : {lower_thr:.4f} USD\n")

# ── 2. Flag outliers and classify direction ───────────────────────────────────
analysis_df["Is_Outlier"]   = (
    (analysis_df["Directional_Error"] > upper_thr) |
    (analysis_df["Directional_Error"] < lower_thr)
)
analysis_df["Outlier_Type"] = "Normal"
analysis_df.loc[analysis_df["Directional_Error"] > upper_thr, "Outlier_Type"] = "Positive Spike"
analysis_df.loc[analysis_df["Directional_Error"] < lower_thr, "Outlier_Type"] = "Negative Drop"

outlier_count = analysis_df["Is_Outlier"].sum()
print(f"Outliers detected: {outlier_count} / {len(analysis_df)} days")

# ── 3. Export full residual history ──────────────────────────────────────────
analysis_df.to_csv("analysis_residuals_2015.csv", index=False)
print("Exported: analysis_residuals_2015.csv")

### 7.2 · Investigation windows

Each outlier date is expanded into an asymmetric research window of **[−3, +7] days**,  
accounting for the typical lag between a geopolitical event and its observable market effect.  
Results are exported as the primary input for the Phase 2 qualitative workbook.

In [ ]:
# ── 1. Absolute residual threshold (2σ on absolute errors) ───────────────────
mean_abs  = analysis_df["Abs_Error"].mean()
std_abs   = analysis_df["Abs_Error"].std()
threshold = mean_abs + 2 * std_abs

print("--- Investigation Period Generation ---")
print(f"Mean absolute error : ${mean_abs:.2f}")
print(f"Std dev             : ${std_abs:.2f}")
print(f"Cutoff (> 2σ)       : ${threshold:.2f}\n")

# ── 2. Isolate outliers and apply asymmetric window ───────────────────────────
outliers = analysis_df[analysis_df["Abs_Error"] > threshold].copy()
outliers["Type"]             = np.where(outliers["Directional_Error"] > 0, "Peak", "Drop")
outliers["Start_Date"]       = outliers["Date"] - pd.Timedelta(days=3)
outliers["End_Date"]         = outliers["Date"] + pd.Timedelta(days=7)

# ── 3. Build and export the investigation table ───────────────────────────────
investigation_df = (
    outliers[["Date", "Actual_Price", "Abs_Error", "Type", "Start_Date", "End_Date"]]
    .rename(columns={
        "Date":         "Event_Date",
        "Actual_Price": "Actual_Price_USD",
        "Abs_Error":    "Error_Value_USD",
    })
    .round({"Actual_Price_USD": 2, "Error_Value_USD": 2})
    .reset_index(drop=True)
)

investigation_df.to_csv("investigation_periods.csv", index=False)

print(f"Outlier events found : {len(investigation_df)}")
print("Exported: investigation_periods.csv")